In [ ]:
import spglib
import os
from ase.io import read
import numpy as np


structure_folders = ["1-PE", "2-PE"]
phase_folders = ["1_c_fm-3m", "2_t_p42_nmc", "3_m_p21_c",
                "4_c-p-43m", "5_t-p4_nbm", "6_m_p21_m",
                "7_o_pbcn", "8_o_p42_mnm", "9_o_i41_amd",
                "1_o1_pca21", "2_o2_pmn21", "3_r_r3"]
doping_type_folders = ["h1z3o8", "h2z2o8", "h3z1o8"]
doping_structure_folders = ["1-doping", "2-doping", "3-doping"]

original_phase_spg = []
doping_unrelax_spg = []
doping_relaxed_spg = []

"""
get the space group of the original phase and the doping structures with and without relaxation,
and save the space group notion to the lists: original_phase_spg, doping_unrelax_spg, doping_relaxed_spg.
the original_phase_spg get from "{structure_folders}/{phase_folders}/CONTCAR", if the directory exists.
the doping_unrelax_spg and dopint_realxed_spg get from the directory of 
"{structure_folders}/{doping_type_folders}/{doping_structure_folders}"'s POSCAR, CONTCAR, respectively.

the space group notion is used the spglib.get_space_group_symbol function, we define a function
to get the space group symbol from the structure file.
"""

def read_structure_file(structure_file):
    if os.path.exists(structure_file):
        with open(structure_file, 'r') as f:
            lines = f.readlines()
        
        scale_factor = float(lines[1].strip())

        lattice_vectors = []
        for i in range(2, 5):
            lattice_vectors.append([float(x) for x in lines[i].strip().split()])

        lattice_vectors = np.array(lattice_vectors) * scale_factor

        atom_types = lines[5].strip().split()
        atom_numbers = [int(x) for x in lines[6].strip().split()]

        coord_type = lines[7].strip()[0].lower()

        positions = []
        num_atoms = sum(atom_numbers)
        for i in range(8, 8 + num_atoms):
            positions.append([float(x) for x in lines[i].strip().split()[:3]])
        positions = np.array(positions)

        if coord_type == 'c' or coord_type == 'k':
            inv_lattice = np.linalg.inv(lattice_vectors)
            positions = np.dot(positions, inv_lattice)

        numbers = []
        for typ, num in zip(atom_types, atom_numbers):
            atomic_number = {
                'H':1, 'He':2, 'Li':3, 'Be':4, 'B':5, 'C':6, 'N':7, 'O':8,
                'F':9, 'Ne':10, 'Na':11, 'Mg':12, 'Al':13, 'Si':14, 'P':15,
                'S':16, 'Cl':17, 'Ar':18, 'K':19, 'Ca':20, 'Sc':21, 'Ti':22,
                'V':23, 'Cr':24, 'Mn':25, 'Fe':26, 'Co':27, 'Ni':28, 'Cu':29,
                'Zn':30, 'Ga':31, 'Ge':32, 'As':33, 'Se':34, 'Br':35, 'Kr':36,
                'Rb':37, 'Sr':38, 'Y':39, 'Zr':40, 'Nb':41, 'Mo':42, 'Tc':43,
                'Ru':44, 'Rh':45, 'Pd':46, 'Ag':47, 'Cd':48, 'In':49, 'Sn':50,
                'Sb':51, 'Te':52, 'I':53, 'Xe':54, 'Cs':55, 'Ba':56, 'La':57,
                'Ce':58, 'Pr':59, 'Nd':60, 'Pm':61, 'Sm':62, 'Eu':63, 'Gd':64,
                'Tb':65, 'Dy':66, 'Ho':67, 'Er':68, 'Tm':69, 'Yb':70, 'Lu':71,
                'Hf':72, 'Ta':73, 'W':74, 'Re':75, 'Os':76, 'Ir':77, 'Pt':78,
                'Au':79, 'Hg':80, 'Tl':81, 'Pb':82, 'Bi':83, 'Po':84, 'At':85,
                'Rn':86, 'Fr':87, 'Ra':88, 'Ac':89, 'Th':90, 'Pa':91,
                'U':92, 'Np':93, 'Pu':94, 'Am':95, 'Cm':96, 'Bk':97,
                'Cf':98, 'Es':99, 'Fm':100, 'Md':101, 'No':102, 'Lr':103,
                'Rf':104, 'Db':105, 'Sg':106, 'Bh':107, 'Hs':108, 'Mt':109,
                'Ds':110, 'Rg':111, 'Cn':112, 'Nh':113, 'Fl':114, 'Mc':115,
                'Lv':116, 'Ts':117, 'Og':118
            }.get(typ, 0)
            numbers.extend([atomic_number] * num)

        return (lattice_vectors, positions, numbers)
    else:
        return None

def get_space_group_symbol(structure_file, symprec=1e-5):


    # if os.path.exists(structure_file):
    #     structure = spglib.get_structure(structure_file)
    #     return spglib.get_space_group_symbol(structure)
    # else:
    #     return None
    if os.path.exists(structure_file):
        cell = read_structure_file(structure_file)
        spg = spglib.get_space_group_symbol(cell, symprec=symprec)
        return spg
    else:
        return None

    
for structure_folder in structure_folders:
    for phase_folder in phase_folders:
        phase_path = os.path.join(structure_folder, phase_folder, "CONTCAR")
        spg = get_space_group_symbol(phase_path)
        if spg:
            original_phase_spg.append(spg)
        else:
            continue
        
        for doping_type_folder in doping_type_folders:
            for doping_structure_folder in doping_structure_folders:
                unrelaxed_path = os.path.join(structure_folder, doping_type_folder, doping_structure_folder, "POSCAR")
                relaxed_path = os.path.join(structure_folder, doping_type_folder, doping_structure_folder, "CONTCAR")
                
                unrelaxed_spg = get_space_group_symbol(unrelaxed_path)
                relaxed_spg = get_space_group_symbol(relaxed_path)
                
                if unrelaxed_spg:
                    doping_unrelax_spg.append(unrelaxed_spg)
                if relaxed_spg:
                    doping_relaxed_spg.append(relaxed_spg)

In [ ]:
import spglib
import os
from ase.io import read
import numpy as np


structure_folders = ["1-PE", "2-PE"]
phase_folders = ["1_c_fm-3m", "2_t_p42_nmc", "3_m_p21_c",
                "4_c-p-43m", "5_t-p4_nbm", "6_m_p21_m",
                "7_o_pbcn", "8_o_p42_mnm", "9_o_i41_amd",
                "1_o1_pca21", "2_o2_pmn21", "3_r_r3"]
doping_type_folders = ["h1z3o8", "h2z2o8", "h3z1o8"]
doping_structure_folders = ["1-doping", "2-doping", "3-doping"]

original_phase_spg = []
doping_unrelax_spg = {}
doping_relaxed_spg = {}

"""
get the space group of the original phase and the doping structures with and without relaxation,
and save the space group notion to the lists: original_phase_spg, doping_unrelax_spg, doping_relaxed_spg.
the original_phase_spg get from "{structure_folders}/{phase_folders}/CONTCAR", if the directory exists.
the doping_unrelax_spg and dopint_realxed_spg get from the directory of 
"{structure_folders}/{doping_type_folders}/{doping_structure_folders}"'s POSCAR, CONTCAR, respectively.

the space group notion is used the spglib.get_space_group_symbol function, we define a function
to get the space group symbol from the structure file.
"""

def read_structure_file(structure_file):
    if os.path.exists(structure_file):
        with open(structure_file, 'r') as f:
            lines = f.readlines()
        
        scale_factor = float(lines[1].strip())

        lattice_vectors = []
        for i in range(2, 5):
            lattice_vectors.append([float(x) for x in lines[i].strip().split()])

        lattice_vectors = np.array(lattice_vectors) * scale_factor

        atom_types = lines[5].strip().split()
        atom_numbers = [int(x) for x in lines[6].strip().split()]

        coord_type = lines[7].strip()[0].lower()

        positions = []
        num_atoms = sum(atom_numbers)
        for i in range(8, 8 + num_atoms):
            positions.append([float(x) for x in lines[i].strip().split()[:3]])
        positions = np.array(positions)

        if coord_type == 'c' or coord_type == 'k':
            inv_lattice = np.linalg.inv(lattice_vectors)
            positions = np.dot(positions, inv_lattice)

        numbers = []
        for typ, num in zip(atom_types, atom_numbers):
            atomic_number = {
                'H':1, 'He':2, 'Li':3, 'Be':4, 'B':5, 'C':6, 'N':7, 'O':8,
                'F':9, 'Ne':10, 'Na':11, 'Mg':12, 'Al':13, 'Si':14, 'P':15,
                'S':16, 'Cl':17, 'Ar':18, 'K':19, 'Ca':20, 'Sc':21, 'Ti':22,
                'V':23, 'Cr':24, 'Mn':25, 'Fe':26, 'Co':27, 'Ni':28, 'Cu':29,
                'Zn':30, 'Ga':31, 'Ge':32, 'As':33, 'Se':34, 'Br':35, 'Kr':36,
                'Rb':37, 'Sr':38, 'Y':39, 'Zr':40, 'Nb':41, 'Mo':42, 'Tc':43,
                'Ru':44, 'Rh':45, 'Pd':46, 'Ag':47, 'Cd':48, 'In':49, 'Sn':50,
                'Sb':51, 'Te':52, 'I':53, 'Xe':54, 'Cs':55, 'Ba':56, 'La':57,
                'Ce':58, 'Pr':59, 'Nd':60, 'Pm':61, 'Sm':62, 'Eu':63, 'Gd':64,
                'Tb':65, 'Dy':66, 'Ho':67, 'Er':68, 'Tm':69, 'Yb':70, 'Lu':71,
                'Hf':72, 'Ta':73, 'W':74, 'Re':75, 'Os':76, 'Ir':77, 'Pt':78,
                'Au':79, 'Hg':80, 'Tl':81, 'Pb':82, 'Bi':83, 'Po':84, 'At':85,
                'Rn':86, 'Fr':87, 'Ra':88, 'Ac':89, 'Th':90, 'Pa':91,
                'U':92, 'Np':93, 'Pu':94, 'Am':95, 'Cm':96, 'Bk':97,
                'Cf':98, 'Es':99, 'Fm':100, 'Md':101, 'No':102, 'Lr':103,
                'Rf':104, 'Db':105, 'Sg':106, 'Bh':107, 'Hs':108, 'Mt':109,
                'Ds':110, 'Rg':111, 'Cn':112, 'Nh':113, 'Fl':114, 'Mc':115,
                'Lv':116, 'Ts':117, 'Og':118
            }.get(typ, 0)
            numbers.extend([atomic_number] * num)

        return (lattice_vectors, positions, numbers)
    else:
        return None

def get_space_group_symbol(structure_file, symprec=1e-5):


    # if os.path.exists(structure_file):
    #     structure = spglib.get_structure(structure_file)
    #     return spglib.get_space_group_symbol(structure)
    # else:
    #     return None
    if os.path.exists(structure_file):
        # structure_file = os.path.abspath
        cell = read_structure_file(structure_file)
        if cell is None:
            return None
        try:
            spg = spglib.get_spacegroup(cell, symprec=symprec).split()[0]
            # print(cell)
            return spg
        except Exception as e:
            print(f"Error getting space group for {structure_file}: {e}")
            return None

    else:
        return None


# spg = get_space_group_symbol("POSCAR")
# #spg = spglib.get_space_group_symbol(structure)
# print(spg)


for structure_folder in structure_folders:
    for phase_folder in phase_folders:
        phase_path = os.path.join(structure_folder, phase_folder, "CONTCAR")
        spg = get_space_group_symbol(phase_path)
        if spg:
            original_phase_spg.append(spg)
        else:
            continue
        
        for doping_type_folder in doping_type_folders:
            
            for doping_structure_folder in doping_structure_folders:
                unrelaxed_path = os.path.join(structure_folder, phase_folder, doping_type_folder, doping_structure_folder, "POSCAR")
                relaxed_path = os.path.join(structure_folder, phase_folder, doping_type_folder, doping_structure_folder, "CONTCAR")
                
                unrelaxed_spg = get_space_group_symbol(unrelaxed_path)
                relaxed_spg = get_space_group_symbol(relaxed_path)
                
                if unrelaxed_spg:
                    doping_unrelax_spg[f"{unrelaxed_path}"]= unrelaxed_spg
                    # doping_unrelax_spg.append(unrelaxed_spg)
                if relaxed_spg:
                    doping_relaxed_spg[f"{relaxed_path}"] = relaxed_spg
                    # doping_relaxed_spg.append(relaxed_spg)

# Print the results
print("Original Phase Space Groups:")
for spg in original_phase_spg:
    print(spg)

print(5*"----")
print("Doping Unrelaxed Space Groups:")
# for spg in doping_unrelax_spg:
#     print(spg)
for k, v in doping_unrelax_spg.items():
    print(f"{k}: {v}")



print(5*"----")
print("Doping Relaxed Space Groups:")
# for spg in doping_relaxed_spg:
#     print(spg)
for k, v in doping_relaxed_spg.items():
    print(f"{k}: {v}")

(array([[5.32324451, 0.        , 0.        ],
       [0.        , 5.09954729, 0.        ],
       [0.        , 0.        , 5.12765889]]), array([[0.03084602, 0.01592433, 0.99817273],
       [0.96915399, 0.48407567, 0.49817273],
       [0.53084601, 0.48407567, 0.99817273],
       [0.46915399, 0.01592433, 0.49817273],
       [0.63484792, 0.6840201 , 0.64203198],
       [0.77324034, 0.21472529, 0.74655529],
       [0.36515205, 0.8159799 , 0.14203199],
       [0.22675964, 0.28527471, 0.24655529],
       [0.13484794, 0.8159799 , 0.64203198],
       [0.27324034, 0.28527471, 0.74655529],
       [0.86515208, 0.6840201 , 0.14203199],
       [0.72675966, 0.21472529, 0.24655529]]), [40, 40, 40, 40, 8, 8, 8, 8, 8, 8, 8, 8])
Pca2_1


In [ ]:
# method 2, read poscar file with ase.io
from ase.io import read
import spglib

atoms = read("POSCAR")
cell = (atoms.cell, atoms.get_scaled_positions(), atoms.get_atomic_numbers())
print(cell)
spg = spglib.get_spacegroup(cell, symprec=1e-1)
transition_operation = spglib.get_symmetry_dataset(cell, symprec=1e-5)
print(transition_operation)
print(spg)

(Cell([5.323244508492787, 5.099547288486947, 5.127658890344059]), array([[0.03084602, 0.01592433, 0.99817273],
       [0.96915399, 0.48407567, 0.49817273],
       [0.53084601, 0.48407567, 0.99817273],
       [0.46915399, 0.01592433, 0.49817273],
       [0.63484792, 0.6840201 , 0.64203198],
       [0.77324034, 0.21472529, 0.74655529],
       [0.36515205, 0.8159799 , 0.14203199],
       [0.22675964, 0.28527471, 0.24655529],
       [0.13484794, 0.8159799 , 0.64203198],
       [0.27324034, 0.28527471, 0.74655529],
       [0.86515208, 0.6840201 , 0.14203199],
       [0.72675966, 0.21472529, 0.24655529]]), array([40, 40, 40, 40,  8,  8,  8,  8,  8,  8,  8,  8]))
SpglibDataset(number=29, hall_number=143, international='Pca2_1', hall='P 2c -2ac', choice='', transformation_matrix=array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]), origin_shift=array([1.  , 0.75, 0.  ]), rotations=array([[[ 1,  0,  0],
        [ 0,  1,  0],
        [ 0,  0,  1]],

       [[-1,  0,  0],
        [ 0, 